##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [18]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [19]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

In [20]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

In [21]:
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

mobilenet_base.trainable = False

In [22]:
mobilenet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224),
    layers.Lambda(preprocess_input),
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)
], name="cifar10_mobilenetv2")

In [23]:
mobilenet_model.summary()

Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_1 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [24]:
print("Total layers in MobileNetV2 backbone:", len(mobilenet_base.layers))

trainable_layers = [layer for layer in mobilenet_base.layers if layer.count_params() > 0]

print("Layers with learnable parameters (depth):", len(trainable_layers))
print("Total parameters:", mobilenet_model.count_params())
print("Trainable parameters:", sum(np.prod(v.shape) for v in mobilenet_model.trainable_weights))
print("Non-trainable parameters:", sum(np.prod(v.shape) for v in mobilenet_model.non_trainable_weights))

Total layers in MobileNetV2 backbone: 154
Layers with learnable parameters (depth): 104
Total parameters: 2270794
Trainable parameters: 12810
Non-trainable parameters: 2257984


In [25]:
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [26]:
history_m = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 78s 105ms/step - accuracy: 0.6035 - loss: 1.1325 - val_accuracy: 0.8008 - val_loss: 0.5803
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 76s 107ms/step - accuracy: 0.7401 - loss: 0.7425 - val_accuracy: 0.8372 - val_loss: 0.4829
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 76s 108ms/step - accuracy: 0.7597 - loss: 0.6923 - val_accuracy: 0.8306 - val_loss: 0.5043


In [27]:
test_loss_m, test_acc_m = mobilenet_model.evaluate(x_test, y_test, verbose=0)

print("MobileNetV2 (frozen) accuracy:", test_acc_m)
print("MobileNetV2 (frozen) loss:", test_loss_m)

MobileNetV2 (frozen) accuracy: 0.8195000290870667
MobileNetV2 (frozen) loss: 0.5273271799087524


In [28]:
mobilenet_base.trainable = True

for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

print("Trainable layers:",
      sum(l.trainable for l in mobilenet_base.layers),
      "/",
      len(mobilenet_base.layers))

Trainable layers: 20 / 154


In [29]:
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [30]:
history_ft_m = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 100s 133ms/step - accuracy: 0.6823 - loss: 0.9207 - val_accuracy: 0.8392 - val_loss: 0.4699
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 92s 130ms/step - accuracy: 0.7667 - loss: 0.6752 - val_accuracy: 0.8462 - val_loss: 0.4392
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 91s 130ms/step - accuracy: 0.7806 - loss: 0.6233 - val_accuracy: 0.8512 - val_loss: 0.4208


In [31]:
test_loss_ft_m, test_acc_ft_m = mobilenet_model.evaluate(x_test, y_test, verbose=0)

print("MobileNetV2 (fine-tuned) accuracy:", test_acc_ft_m)
print("MobileNetV2 (fine-tuned) loss:", test_loss_ft_m)

MobileNetV2 (fine-tuned) accuracy: 0.8468999862670898
MobileNetV2 (fine-tuned) loss: 0.4488881230354309


1. Which model achieved the highest accuracy?

ResNet50V2 achieved the highest test accuracy at 91.62%, followed by MobileNetV2 at 84.68%, and the custom CNN at 70.28%. 
This result shows that deeper pretrained architectures significantly outperform a custom CNN trained from scratch. ResNet50V2 provided the best feature representation and generalization performance on CIFAR-10.

2. Which model trained faster?

The custom CNN trained the fastest, since it has the smallest number of parameters and a simpler architecture.

Between the pretrained models, MobileNetV2 trained faster than ResNet50V2. MobileNetV2 is specifically designed to be lightweight and computationally efficient, while ResNet50V2 is deeper and contains significantly more parameters, making it computationally heavier.

3. How might the architecture explain the differences?

The performance differences can be explained by architectural design:

Custom CNN:
A shallow network trained from scratch. It lacks pretrained knowledge and has limited feature extraction capability, which explains its lower accuracy.

MobileNetV2:
Uses depthwise separable convolutions and inverted residual blocks. This design reduces parameters and computation while maintaining strong feature extraction ability. It achieves good accuracy with high efficiency.

ResNet50V2:
A deep residual network with skip connections that help prevent vanishing gradients and allow very deep feature learning. Its large depth and parameter count enable it to learn complex hierarchical image features, which explains its superior accuracy.

Overall, deeper architectures with residual connections (ResNet50V2) provide higher accuracy, while lightweight architectures (MobileNetV2) offer a balance between speed and performance.